### Fase 1: Transformando Canciones en "Canastas"
El algoritmo Apriori no lee DataFrames normales; necesita una lista de transacciones. 
En nuestro caso, cada **Canción** es una transacción, y los **Artistas** que participan en ella son los artículos de la canasta.

Debido a que en la columna de artistas solo se muestra el primer nombre y se omiten los colaboradores, debemos de usar la columna donde se puedan encontrar los artistas. Esta columna es la del nombre de la cancion (Track). En lugar de usar split(,), usaremos las palabras 'feat.' e '&' y si las canciones los contienen, se agregaran a la lista de nombres ademas del de la columna Artist. 

In [18]:
# Instalar si es necesario: !pip install mlxtend
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import re

# 1. INGESTA DE DATOS
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
df_spotify = pd.read_csv(url, encoding='latin-1')

# 2. LIMPIEZA: Eliminar nulos de la columna de artistas
df_spotify = df_spotify.dropna(subset=['Track', 'Artist'])

#funcion extra para detectar artistas colaboradores en el nombre de la cancion
def extraer_artistas(fila):
    artistas = [fila['Artist'].strip()]
    m = re.search(r'feat\.\s*(.+)', fila['Track'], flags=re.IGNORECASE) #se usa el raw string (no ignora caracteres) y espacios en blanco, permite encontrar uno o +
    if m:
        invitados = m.group(1).strip(' )]')
        artistas += [a.strip() for a in re.split(r'&|,', invitados)] 
    return artistas

# 3. TRANSFORMACIÓN: Separar los artistas que colaboran en cada canción
# Asumimos que las colaboraciones están separadas por coma (ej. "Bad Bunny, Feid")
# Convertimos esa columna en una verdadera lista de Python
lista_colaboraciones = df_spotify.apply(extraer_artistas, axis=1).tolist()

# Muestra rápida de cómo se ven nuestras "canastas"
# print("Muestra de las primeras 5 colaboraciones:")
# for i in range(5):
#     print(f"Canción {i+1}: {lista_colaboraciones[i]}")

print("Canciones con 2+ artistas:", sum(len(x) > 1 for x in lista_colaboraciones))
print("Muestra de las primeras 5 colaboraciones")
print([artist for artist in lista_colaboraciones if len(artist) >1][:5]) 


Canciones con 2+ artistas: 470
Muestra de las primeras 5 colaboraciones
[['Central Cee', 'Lil Baby'], ['Post Malone', 'Morgan Wallen'], ['Taylor Swift', 'Post Malone'], ['Sam Smith', 'Kim Petras'], ['JUVENILE', 'Lil Wayne', 'Mannie Fresh']]


In [19]:
# Utiliza TransactionEncoder para convertir 'lista_colaboraciones' en una matriz booleana (One-Hot)
te = TransactionEncoder()
matriz_bool = te.fit_transform(lista_colaboraciones)
# Guárdala en un DataFrame llamado 'df_transacciones'
df_transacciones = pd.DataFrame(matriz_bool, columns=te.columns_)


### Fase 2: Aplicando Apriori
Ahora que tenemos la matriz con ceros y unos, extraigamos los conjuntos de artistas que colaboran frecuentemente.
Debido a que hay miles de artistas solistas, las colaboraciones son estadísticamente raras. Usaremos un `min_support` muy bajo para capturar las colaboraciones.

In [51]:
# --- RETO PARA EL ALUMNO AQUÍ ---

# 1. Encuentra los Itemsets Frecuentes usando un soporte mínimo del 0.005 (0.5% de las canciones)
itemsets_frecuentes = apriori(df_transacciones, min_support=0.0006, use_colnames=True)

# 2. Genera las Reglas de Asociación filtrando por la métrica 'lift' con un mínimo de 1.2
reglas_musicales = association_rules(itemsets_frecuentes, metric="lift", min_threshold=1.2)

# 3. Muestra las reglas resultantes ordenadas por 'confidence' de mayor a menor
display(reglas_musicales.sort_values(by='confidence', ascending=False)[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

,antecedents,consequents,support,confidence,lift
5,(PARTYNEXTDOOR),(Drake),0.000653,0.750000,41.026786
9,(Sexyy Red),(Drake),0.000653,0.500000,27.351190
3,(DJ Khaled),(Drake),0.000653,0.428571,23.443878
11,(Young Thug),(Gunna),0.000653,0.250000,57.437500
13,(Lil Durk),(Lil Baby),0.000653,0.214286,30.770089
14,(Lil Uzi Vert),(Nicki Minaj),0.000653,0.176471,42.678019
15,(Nicki Minaj),(Lil Uzi Vert),0.000653,0.157895,42.678019
10,(Gunna),(Young Thug),0.000653,0.150000,57.437500
7,(SZA),(Drake),0.000653,0.142857,7.814626
1,(21 Savage),(Drake),0.000653,0.115385,6.311813


Decidi bajar el porcentaje de soporte ya qeu el 0.005 no mostraba ninguna regla que cayera bajo ese porcentaje. Con el 0.0005 ya hay colaboraciones que tienen mas apariciones.

### 🧠 Fase 3: Interpretación de Negocio (El Entregable)

Como Científico de Datos, no basta con entregar una tabla. Debes explicarle los hallazgos al Director Creativo. Revisa tu DataFrame `reglas_musicales` y responde las siguientes 4 preguntas:

**1. Análisis de Soporte (El Volumen):**
Menciona a los dos artistas que conforman la regla con el mayor **Soporte**. ¿Qué significa este número en términos simples para el negocio?

> **Tu Respuesta:** Hay 16 reglas en total, todas con el mismo soporte, 0.000653, así que hay un empate. Algunas de las colaboraciones que aparecen son Drake y 21 Savage, Drake y SZA, Young Thug y Gunna, y Lil Durk y Lil Baby. En términos de negocio, el soporte es el volumen: qué porcentaje del catálogo ocupa esa colaboración. Aquí es el 0.065%, es decir, la colaboración existe pero es muy poco frecuente y se puede concluir que ninguna pareja destaca por volumen.

**2. Análisis de Confianza (La Dirección):**
Busca una regla que tenga una **Confianza** mayor al 0.5 (50%). Traduce esta regla matemática a una frase que el director pueda entender (Ej. "Si contratamos al artista A, existe un X% de probabilidad de que el artista B también deba estar en el track para que sea el éxito que registramos").

> **Tu Respuesta:** PARTYNEXTDOOR y Drake tienen juntos una confianza de 0.75, lo que significa que si hacemos un contrato con PARTYNEXTDOOR, existe un 75% de probabilidad de que Drake tambien deba estar en la cancion para que alcance el exito que queremos.

**3. Análisis de Lift (El Impulso):**
Busca la regla con el **Lift** más alto de toda tu tabla. 
¿Por qué esta métrica es más confiable que el Soporte para decidir si estos dos artistas *realmente* tienen química musical o si solo fue una coincidencia estadística?

> **Tu Respuesta:** Gunna y Yung Thug son la colaboración con mayor lift (57.43), esta metrica es mas confiable que el soporte porque éste solo cuenta cuantas veces aparecen juntos, sin considerar qué tan populares son por separado. mientras que el lift compara cuántas veces colaboran realmente contra cuántas veces se esperaría que coincidieran por azar si fueran independientes. 
**4. Toma de Decisión:**
Basado puramente en las reglas matemáticas que descubriste, si la disquera solo tiene presupuesto para juntar a dos artistas en el estudio la próxima semana, ¿qué pareja elegirías y por qué? (Justifica usando las 3 métricas de oro).

> **Tu Respuesta:** Si tuviera que escoger una colaboracion a partir de estas metricas, eligiria a Drake y PARTYNEXTDOOR, porque como vimos, su confianza es la mas elevada y su lift no es el mas elevado de la tabla pero es de los más altos, por lo que con esta proporción, lograríamos conseguir el exito deseado.